# VQA competition runner (Colab / Omnicampus)

Thin runner: all logic lives in `src/`. This notebook clones the repo, prepares the
data, trains a model, writes `submission.npy` + `model.pt`, and zips the submission.

**Reproducibility note:** the grader runs this notebook top-to-bottom, so the cloned
repo must be public and pinned to a commit (set `REPO_URL` / `COMMIT` below). If the
repo stays private, replace the clone cell with the self-contained code instead.

Strategy: run the safety-net config first (`resnet50_concat`), submit early, then rerun
with `vit_bert_attn` for the improved score.

In [ ]:
# 1. Clone the repo (pin a commit for reproducibility) and install deps
REPO_URL = "https://github.com/<your-user>/multimodal-vqa.git"
COMMIT = ""  # e.g. "a1b2c3d"; leave empty to use the default branch head
!git clone $REPO_URL
%cd multimodal-vqa
if COMMIT:
    !git checkout $COMMIT
!pip install -q -r requirements.txt

In [ ]:
# 2a. Colab: mount Drive, copy data.zip (12GB, prepared by data_download_VQA.ipynb), unzip.
#     Produces data/train.json, data/valid.json, data/train/, data/valid/ (config root: data)
from google.colab import drive
drive.mount('/content/drive')
!cp "/content/drive/MyDrive/data.zip" .
!unzip -q data.zip
!ls data

In [ ]:
# 2b. Omnicampus alternative: data is fetched via gsutil; point root at its directory.
#     (Skip cell 2a if you use this path.)
# !gsutil -m cp -r gs://<bucket>/VQA/data ./data

In [ ]:
# 3. Run 1 (safety net): pretrained ResNet50 + one-hot + concat. Aim: comfortably >= 49.9%.
!python -m src.train --config configs/resnet50_concat.yaml

In [ ]:
# 4. Write submission.npy + model.pt from the best checkpoint
!python -m src.predict --config configs/resnet50_concat.yaml \
  --ckpt experiments/resnet50_concat/best.pt --out submission/submission.npy

In [ ]:
# 5. Verify the submission format before packaging
import numpy as np, os
a = np.load('submission/submission.npy')
print('shape', a.shape, 'dtype', a.dtype)
assert a.shape == (4969,), 'test set has 4969 samples'
assert a.dtype.kind in ('U', 'S', 'O'), 'answers must be strings'
print('submission.npy OK')

In [ ]:
# 6. Zip the three required artifacts and check the 4.5GB limit
from zipfile import ZipFile
NOTEBOOK = '/content/drive/MyDrive/Colab Notebooks/colab_runner.ipynb'  # adjust to this notebook's path
with ZipFile('submission.zip', 'w') as zf:
    zf.write('submission/submission.npy', arcname='submission.npy')
    zf.write('submission/model.pt', arcname='model.pt')
    zf.write(NOTEBOOK, arcname='colab_runner.ipynb')
size_gb = os.path.getsize('submission.zip') / 1e9
print(f'submission.zip = {size_gb:.2f} GB')
assert size_gb <= 4.5, 'zip exceeds 4.5GB limit'

## Next runs (improved score / ablations)

Rerun cells 3-6 with a different config (no code changes):
- `configs/vit_bert_attn.yaml` — ViT + BERT + cross-attention + soft labels (target ~60%).
- Ablations for the report: swap `fusion.type` (concat vs cross_attention),
  `image_encoder.pretrained` (scratch vs pretrained), `train.soft_label` (hard vs soft).

Keep the best `submission.npy` / `model.pt` locally and re-submit the best right before the
deadline — only the last submission is graded, and it must stay >= 49.9%.